# Compare Training Experiments

This notebook compares training runs across different approaches:
- Baseline PF-CNN
- PF-CNN + MDA-DMC augmentation
- CLSR-AMC (Contrastive Learning with Self-Reconstruction)

In [ ]:
import sys
from pathlib import Path

src_path = Path("../src")
if src_path.exists():
    sys.path.insert(0, str(src_path.resolve()))

import matplotlib.pyplot as plt

from robust_amc.training import (
    load_history_from_checkpoint,
    plot_multi_run_comparison,
    plot_loss_component_breakdown,
    plot_comprehensive_diagnostic,
    analyze_training_diagnostics,
)

## 1. Load Training Histories

In [ ]:
CHECKPOINTS_DIR = Path("../checkpoints")

histories = {}

# Load each checkpoint
checkpoint_paths = {
    "Baseline": CHECKPOINTS_DIR / "baseline_2016" / "best_model.pt",
    "MDA-DMC": CHECKPOINTS_DIR / "mda_dmc_2016" / "best_model.pt",
    "CLSR-AMC": CHECKPOINTS_DIR / "clsr_amc_2016" / "best_model.pt",
}

for name, path in checkpoint_paths.items():
    history = load_history_from_checkpoint(path)
    if history:
        histories[name] = history
        n_epochs = len(history.get("train_loss", []))
        print(f"Loaded: {name} ({n_epochs} epochs)")
    else:
        print(f"Not found: {name} at {path}")

if not histories:
    print("\nNo checkpoints found! Run training scripts first.")

## 2. Validation Accuracy Comparison

In [ ]:
if histories:
    fig = plot_multi_run_comparison(histories, metric="val_acc", title="Validation Accuracy")
    plt.show()

## 3. Training Loss Comparison

In [ ]:
if histories:
    fig = plot_multi_run_comparison(histories, metric="train_loss", title="Training Loss")
    plt.show()

## 4. Comprehensive Diagnostic

In [ ]:
if histories:
    fig = plot_comprehensive_diagnostic(histories)
    plt.show()

## 5. CLSR-AMC Loss Breakdown

In [ ]:
if "CLSR-AMC" in histories:
    clsr_history = histories["CLSR-AMC"]
    if "train_contrastive" in clsr_history:
        fig = plot_loss_component_breakdown(clsr_history)
        plt.show()
    else:
        print("CLSR-AMC checkpoint doesn't have loss component breakdown")

## 6. Automated Diagnostics

In [ ]:
if histories:
    diagnostics = analyze_training_diagnostics(histories)
    
    if diagnostics["warnings"]:
        print("Warnings:")
        for warning in diagnostics["warnings"]:
            print(f"  - {warning}")
    
    print("\nSummary:")
    for info in diagnostics["info"]:
        print(f"  - {info}")

## 7. Summary Table

In [ ]:
if histories:
    print(f"{'Model':<15} {'Best Val Acc':>12} {'Best Epoch':>12}")
    print("-" * 41)
    for name, history in histories.items():
        best_acc = history.get("best_val_acc", "N/A")
        best_epoch = history.get("best_epoch", "N/A")
        if isinstance(best_acc, float):
            best_acc = f"{best_acc:.4f}"
        print(f"{name:<15} {best_acc:>12} {best_epoch:>12}")